<a href="https://colab.research.google.com/github/muajnstu/Portfolio-Management-using-RL/blob/master/DSE_Implementation_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DSE 14 — one seed (42), port of the uploaded DJIA29 core

This notebook carries the uploaded DJIA29 engine's `05_train_cnn.py`, `policy_network.py`, `portfolio_env.py`, `08_train_agents.py`, `meta_gate.py`, `09_neural_gate_study.py` and Phase-4 routing code. The **CNN temporal/attention architecture, CNN feature extractor, five DRL agent types, robust expert calibration, soft-utility gate supervision, four gate candidate architectures, validation selection, and Top-1/2/3/dense routing** come from those source modules. It runs only seed 42. The first full-seed report includes Buy and Hold, Equal Weight and MVO. No other seeds are launched.

**DSE adapters are unavoidable and disclosed:** 14 ticker slots and DSE sessions, eight input channels including freshness, four DSE-local HMM inputs instead of U.S. macro data, candidate 10:1 continuity factors for nine events, target masking where a genuine quote is unavailable, zero-return stale marks until quote resumption, frozen stock weights when execution lacks a quote, BDT 1,000,000 initial AUM, zero cash yield, and a provisional 10 bp per-side commission scenario with unverified spread/impact disabled. The shared model, expert, gate, and routing modules are reused from the DJIA bundle. Results are a **development transfer test** until corporate actions, provisional sessions and costs are verified.

Run each cell in order. The first stages verify data and shapes, the source CNN fit, one masked environment step, all five constructors, and an isolated **short smoke run**. The final cells train the complete DJIA free-mode seed (20,000 steps per expert), then fit/select the gate and route and compare with benchmarks. A smoke checkpoint is never eligible for the full-run manifest. The full run has **not** been executed in this notebook artifact; local short-path tests passed with the provided prepared DSE data. If a validation cell fails, stop there and share its traceback.


In [ ]:
%pip -q install 'torch>=2.5,<3' 'stable-baselines3>=2.4,<3' 'gymnasium>=1.0,<1.3' 'hmmlearn>=0.3,<1' 'scikit-learn>=1.3' 'scipy>=1.10' 'joblib>=1.3'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 14.9 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import os,json,io,base64,hashlib,zipfile,shutil,subprocess,sys
from google.colab import drive
drive.mount('/content/drive')
PROJECT=Path(os.environ.get('DSE_PROJECT_ROOT','/content/drive/MyDrive/DSE_2004_2025_Preparation'))
SOURCE=Path(os.environ.get('DSE_14_OUTPUT',str(PROJECT/'prepared_14_Annual95_2004_Apr2025')))
FOLDER_ID='1t2MLPWfwyRswWDEDNAGvQJWE7y9Z8YU_'
REQUIRED=['audit_manifest.json','availability.csv.gz','raw_close.csv.gz','raw_valid_observed.csv.gz']
def existing_source():
    if all((SOURCE/f).is_file() for f in REQUIRED):return SOURCE
    from google.colab import auth
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    auth.authenticate_user()
    credentials,_=google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
    service=build('drive','v3',credentials=credentials,cache_discovery=False)
    local=PROJECT/'downloaded_prepared_14';local.mkdir(parents=True,exist_ok=True)
    for name in REQUIRED:
        q=f"'{FOLDER_ID}' in parents and name = '{name}' and trashed = false"
        rows=service.files().list(q=q,fields='files(id,name,size)',supportsAllDrives=True,
            includeItemsFromAllDrives=True,page_size=100).execute().get('files',[])
        if len(rows)!=1:raise FileNotFoundError(f'{name}: expected exactly one source file; found {len(rows)}')
        dest=local/name;size=int(rows[0]['size'])
        if not dest.exists() or dest.stat().st_size!=size:
            request=service.files().get_media(fileId=rows[0]['id'],supportsAllDrives=True)
            with dest.open('wb') as stream:
                downloader=MediaIoBaseDownload(stream,request);done=False
                while not done:_,done=downloader.next_chunk()
    return local
SOURCE=existing_source()
RUN=Path(os.environ.get('DSE_DJIA_CORE_RUN',str(PROJECT/'DSE14_DJIA_CORE_seed42_v1')))
CODE=RUN/'code';ENGINE=CODE/'engine';RUN.mkdir(parents=True,exist_ok=True)
ENV=os.environ.copy();ENV.update(FIXED29_PROJECT_ROOT=str(RUN),FIXED29_TEST_END='2025-04-08',
    DSE_PREPARED_14=str(SOURCE),MARKET_EXPERIMENT_SEED='42',MARKET_GATE_SEED='42',
    FIXED29_EXTENSION_SEED='42',FIXED29_EXTENSION_MODE='free',OMP_NUM_THREADS='2',MKL_NUM_THREADS='2')
print('Prepared input:',SOURCE,'Full seed output:',RUN)


Mounted at /content/drive
Prepared input: /content/drive/MyDrive/DSE_2004_2025_Preparation/prepared_14_Annual95_2004_Apr2025 Full seed output: /content/drive/MyDrive/DSE_2004_2025_Preparation/DSE14_DJIA_CORE_seed42_v1


In [ ]:
BUNDLE_SHA256='acc3554dcaf90366b54e351aaeb5edc8dad46ee4d6476d2acdea32bd20417800'
BUNDLE_B64='...'
blob=base64.b64decode(BUNDLE_B64)
assert hashlib.sha256(blob).hexdigest()==BUNDLE_SHA256
CODE.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(blob)) as archive:
    for item in archive.infolist():
        part=Path(item.filename)
        assert not part.is_absolute() and '..' not in part.parts
        assert part.parts[0]=='engine'
    archive.extractall(CODE)
print('Verified DSE-patched DJIA engine bundle:',BUNDLE_SHA256,'modules:',len(list(ENGINE.glob('*.py'))))
def run_script(name,*args,env=None):
    config=ENV.copy();config.update(env or {})
    command=[sys.executable,name,*args]
    print('Running:',name,*args,flush=True)
    subprocess.run(command,cwd=ENGINE,env=config,check=True)


Verified DSE-patched DJIA engine bundle: acc3554dcaf90366b54e351aaeb5edc8dad46ee4d6476d2acdea32bd20417800 modules: 23


## 1. Build the DSE data adapter and audit stage boundaries

In [ ]:
run_script('dse_prepare_samples.py')
import numpy as np,pandas as pd
audit=json.loads((RUN/'data/samples/dse_mask_audit.json').read_text())
assert all(audit[k]['samples']>0 for k in ('train','validation','test'))
for split,(start,end) in {'train':('2004-01-01','2009-12-31'),
    'validation':('2010-01-01','2015-12-31'),'test':('2016-01-01','2025-04-08')}.items():
    with np.load(RUN/'data/samples'/f'{split}.npz') as d:
        assert d['X'].shape[1:]==(14,60,8)
        assert d['tradable_at_execution'].shape==(len(d['X']),14)
        assert np.isfinite(d['daily_asset_returns']).all()
        assert all(pd.Timestamp(start)<=pd.Timestamp(str(z))<=pd.Timestamp(end) for z in d['exit_dates'])
        assert (d['target_excess_return_zscore'][~d['target_mask']]==0).all()
print('DATA PREFLIGHT PASS',audit)

Running: dse_prepare_samples.py
DATA PREFLIGHT PASS {'train': {'samples': 1373, 'first': '2004-03-24', 'last': '2009-12-30', 'unavailable_execution_stock_days': 258}, 'validation': {'samples': 1415, 'first': '2010-01-03', 'last': '2015-12-31', 'unavailable_execution_stock_days': 134}, 'test': {'samples': 2160, 'first': '2016-01-03', 'last': '2025-04-08', 'unavailable_execution_stock_days': 196}, 'rejected_boundary_samples': {'train': 21, 'validation': 21, 'test': 0}, 'corporate_actions_verified': False, 'costs_verified': False, 'missing_rule': 'stale valuation, zero daily marked return during gap, quote on resumption, no trade without fresh execution quote'}


## 2. Fit the source CNN in free mode and create source-format RL states

In [ ]:
run_script('05_train_cnn.py','--quick','--seed','42')
run_script('06_precompute_states.py')
for split in ('train','validation','test'):
    with np.load(RUN/'data/states'/f'{split}.npz') as d:
        assert d['stock_embeddings'].shape[1:]==(14,64)
        assert d['tradable_at_execution'].shape==(len(d['decision_dates']),14)
print('SOURCE CNN AND STATE PASS')


Running: 05_train_cnn.py --quick --seed 42
Running: 06_precompute_states.py
SOURCE CNN AND STATE PASS


## 3. One masked environment step and all five reference agent constructors

In [ ]:
import sys
sys.path.insert(0,str(ENGINE))
os.environ.update({k:ENV[k] for k in ('FIXED29_PROJECT_ROOT','FIXED29_TEST_END','MARKET_EXPERIMENT_SEED')})
from research_config import DATA_ROOT
from portfolio_env import PortfolioEnv
from importlib import import_module
from stable_baselines3 import DDPG,TD3,SAC,PPO,A2C
from policy_network import CNNAlphaPolicyExtractor
from stable_baselines3.common.noise import NormalActionNoise
train_file=DATA_ROOT/'states/train.npz';env=PortfolioEnv(train_file)
obs,_=env.reset(seed=42)
assert obs.shape==(14*69+86,) and env.action_space.shape==(16,)
missing_idx=int(np.flatnonzero(~env.tradable_at_execution.all(axis=1))[0])
missing=~env.tradable_at_execution[missing_idx]
old=np.zeros(env.n_assets);old[:-1][missing]=.05;old[-1]=1-old[:-1].sum()
desired=np.r_[np.full(14,1/14),0.]
executed,_=env._apply_liquidity_cap_between(old,desired,missing_idx,1.)
assert np.allclose(executed[:-1][missing],old[:-1][missing])
obs,reward,done,truncated,info=env.step(env.action_space.sample())
assert np.isfinite(reward)
from importlib.util import spec_from_file_location,module_from_spec
spec=spec_from_file_location('dse_agent_training',ENGINE/'08_train_agents.py')
agent_module=module_from_spec(spec);spec.loader.exec_module(agent_module)
for name in ('DDPG','TD3','SAC','PPO','A2C'):
    model,steps=agent_module.build_model(name,42,free_colab=True,device='cpu')
    extractor = (model.policy.features_extractor if name in ('PPO','A2C')
                 else model.policy.actor.features_extractor)
    assert isinstance(extractor,CNNAlphaPolicyExtractor), (name,type(extractor))
    del model
print('ONE STEP, FROZEN SLOT, AND FIVE CONSTRUCTORS PASS')


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


ONE STEP, FROZEN SLOT, AND FIVE CONSTRUCTORS PASS


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 4. Isolated short training, calibration, gate, route and benchmark smoke test

In [ ]:
SMOKE=Path('/content/dse14_core_smoke_seed42')
if SMOKE.exists():shutil.rmtree(SMOKE)
state_dir=SMOKE/'data/states';state_dir.mkdir(parents=True)
for name in ('train','validation','test'):
    shutil.copy2(RUN/'data/states'/f'{name}.npz',state_dir/f'{name}.npz')
smoke_env={'FIXED29_PROJECT_ROOT':str(SMOKE),'DSE_SMOKE_ONLY':'1'}
run_script('08_train_agents.py','--mode','free','--agent','all',env=smoke_env)
run_script('09_neural_gate_study.py','--mode','free','--train-select',env=smoke_env)
run_script('09_neural_gate_study.py','--mode','free','--evaluate',env=smoke_env)
selected=json.loads((SMOKE/'models/active_dir_agents/neural_gate_candidates/selected_gate.json').read_text())['selected_architecture']
run_script('phase2_9_three_gates.py','--phase','4',env={**smoke_env,'FIXED29_PRIMARY_GATE':selected})
run_script('dse_benchmarks.py',env=smoke_env)
assert len(pd.read_csv(SMOKE/'results/active_dir_reward/dse_benchmarks/test_metrics.csv'))==3
assert len(pd.read_csv(SMOKE/'phase2_9_extension/phase4_routing/test_routing_metrics.csv'))>=4
print('SHORT END-TO-END TEST PASSED. No smoke checkpoint is in the full run:',RUN)


Running: 08_train_agents.py --mode free --agent all
Running: 09_neural_gate_study.py --mode free --train-select
Running: 09_neural_gate_study.py --mode free --evaluate
Running: phase2_9_three_gates.py --phase 4
Running: dse_benchmarks.py
SHORT END-TO-END TEST PASSED. No smoke checkpoint is in the full run: /content/drive/MyDrive/DSE_2004_2025_Preparation/DSE14_DJIA_CORE_seed42_v1


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 5. Full seed 42 — all five independent experts and calibration checkpoints

In [ ]:
assert (SMOKE/'phase2_9_extension/phase4_routing/test_routing_metrics.csv').exists()
run_script('08_train_agents.py','--mode','free','--agent','all')
manifest=json.loads((RUN/'models/active_dir_agents/agents/manifest.json').read_text())
names={x['agent'] for x in manifest['models'] if x.get('run_mode')=='free_colab' and x.get('seed')==42}
assert names=={'DDPG','TD3','SAC','PPO','A2C'}
print('Full seed 42 calibrated experts complete; smoke models were not reused.')


Running: 08_train_agents.py --mode free --agent all
Full seed 42 calibrated experts complete; smoke models were not reused.


## 6. Full seed 42 — original soft-utility gate and pretest architecture selection

In [ ]:
run_script('09_neural_gate_study.py','--mode','free','--train-select')
selected=json.loads((RUN/'models/active_dir_agents/neural_gate_candidates/selected_gate.json').read_text())['selected_architecture']
print('Selected pretest architecture:',selected)
run_script('09_neural_gate_study.py','--mode','free','--evaluate')


Running: 09_neural_gate_study.py --mode free --train-select
Selected pretest architecture: GRU
Running: 09_neural_gate_study.py --mode free --evaluate


## 7. Full seed 42 — validation-selected route and shared-engine benchmarks

In [ ]:
run_script('phase2_9_three_gates.py','--phase','4',env={'FIXED29_PRIMARY_GATE':selected})
run_script('dse_benchmarks.py')
route=pd.read_csv(RUN/'phase2_9_extension/phase4_routing/test_routing_metrics.csv')
assert route.Selected_PreOOS.astype(bool).sum()==1
benchmark=pd.read_csv(RUN/'results/active_dir_reward/dse_benchmarks/test_metrics.csv')
assert set(benchmark.Strategy)=={'Buy and Hold','Equal Weight','MVO'}
primary=route[route.Selected_PreOOS.astype(bool)][['Spec','annualized_return','sharpe_ratio','sortino_ratio','maximum_drawdown','final_portfolio_value']].rename(columns={'Spec':'Strategy'})
cols=list(primary.columns)
print(pd.concat([primary,benchmark[cols]],ignore_index=True).to_string(index=False))
status={'seed':42,'source':'uploaded DJIA29 engine with DSE data/mask adapter',
  'full_seed_42_completed':True,'benchmark_names':benchmark.Strategy.tolist(),
  'selected_architecture':selected,'selected_route':json.loads((RUN/'phase2_9_extension/phase4_routing/selected_route.json').read_text()),
  'corporate_actions_verified':False,'provisional_sessions_verified':False,'costs_verified':False,
  'ready_for_thesis_results':False,'other_seeds_started':False}
(RUN/'seed42_status.json').write_text(json.dumps(status,indent=2))
print(json.dumps(status,indent=2))


Running: phase2_9_three_gates.py --phase 4
Running: dse_benchmarks.py
        Strategy  annualized_return  sharpe_ratio  sortino_ratio  maximum_drawdown  final_portfolio_value
{"mode": "top2"}           0.022994      0.216535       0.329164          0.474208               1.215142
    Buy and Hold          -0.006215      0.067612       0.100207          0.495906               0.947961
    Equal Weight           0.027322      0.238906       0.362847          0.478135               1.259921
             MVO           0.001567      0.086457       0.129512          0.427789               1.013509
{
  "seed": 42,
  "source": "uploaded DJIA29 engine with DSE data/mask adapter",
  "full_seed_42_completed": true,
  "benchmark_names": [
    "Buy and Hold",
    "Equal Weight",
    "MVO"
  ],
  "selected_architecture": "GRU",
  "selected_route": {
    "mode": "top2"
  },
  "corporate_actions_verified": false,
  "provisional_sessions_verified": false,
  "costs_verified": false,
  "ready_for_thesis

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
